# Twitter Scraper 

### Luiz Verheyen 

In [1]:
# imports

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import re
import time
import pandas as pd
from datetime import datetime
import os
from dotenv import load_dotenv

In [2]:
# zorgen dat ik de env credentials kan gebruiken
load_dotenv() 

True

## Credentials needed for login
- email
- username
- password

In [3]:
my_twitter_email = os.getenv("my_twitter_email")
my_twitter_username = os.getenv("my_twitter_username")
my_twitter_password = os.getenv("my_twitter_password")

### options for the scraping configuration

In [4]:
options = uc.ChromeOptions() #initializing
options.add_argument("--start-maximized") # start screen volledig
options.add_argument("--disable-notifications") # disable alle notificaties die zouden binnenkomen

In [5]:
driver = uc.Chrome(version_main=145, options=options)
time.sleep(2)

In [6]:
# Ga naar loginpagina
driver.get("https://twitter.com/login")
time.sleep(2)  # wacht tot de pagina volledig geladen is

In [7]:
try:
    email_input = driver.find_element(By.NAME, "text")
    email_input.send_keys(my_twitter_email)
    time.sleep(1)
    email_input.send_keys(Keys.ENTER)
    time.sleep(3)
except:
    print("Geen Username input gevonden of al ingevuld.")
    
# 2 factor authentication
try:
    username_input = driver.find_element(By.NAME, "text")
    username_input.send_keys(my_twitter_username)
    time.sleep(1)
    username_input.send_keys(Keys.ENTER)
    time.sleep(3)
except:
    print("no 2 factor authentication found or not needed.")
    
# password in tikken
try:
    password_input = driver.find_element(By.NAME, "password")
    password_input.send_keys(my_twitter_password)
    time.sleep(1)
    password_input.send_keys(Keys.ENTER)
    time.sleep(3)
except:
    print("Geen wachtwoord invoer vereist of al ingelogd.")

no 2 factor authentication found or not needed.


In [8]:
def twitter_handle(since, until, username):
    driver.get(f"https://x.com/search?q=from%3A{username}%20since%3A{since}%20until%3A{until}&f=live")
    time.sleep(3)  # wachten tot pagina laadt
    # zorg dat cookies worden ge accepteert indien nodig:
    try:
        cookie_button = driver.find_element(By.XPATH, '//button[contains(., "Accept all cookies")]')
        cookie_button.click()
        print("Cookies geaccepteerd.")
        # time.sleep(2)
    except:
        print("Geen cookie-wall gevonden of al geaccepteerd.")

In [9]:
def human_scroll(driver, total_scroll=3000, step=300, pause=1):
    scrolled = 0
    while scrolled < total_scroll:
        driver.execute_script(f"window.scrollBy(0, {step});")
        scrolled += step
        time.sleep(pause)

In [10]:
def save_to_csv(tweets_data):
    df = pd.DataFrame(tweets_data, columns=["Date", "Username", "Content", "Replies", "Reposts", "Likes", "Bookmarks", "Views"])
    df['Date'] = df['Date'].dt.strftime('%Y-%m-%d %H:%M')
    df['Time'] = pd.to_datetime(df['Date']).dt.time
    df['Date'] = pd.to_datetime(df['Date']).dt.date
    print(f"{len(df)} tweets opgeslagen!")
    return df

In [11]:
# usernames_we_wanna_scrape = ["tim_cook", "elonmusk"]
usernames_we_wanna_scrape = ["elonmusk"]
tweets_data = []

In [12]:
from datetime import datetime
import time
import re
import os
import pandas as pd
from selenium.webdriver.common.by import By

# Instellingen
since = "2025-12-31"
scroll_pause = 2
MAX_RETRY = 3

for user in usernames_we_wanna_scrape:
    print(f"\n--- Start scraping {user} ---")

    until = datetime.today().strftime("%Y-%m-%d")
    until = "2026-01-22"
    tweets_ids = set()
    tweets_data = []
    retry = 0
    stop_scraping = False

    twitter_handle(username=user, since=since, until=until)

    while not stop_scraping:

        refresh_needed = False
        tweets_before = len(tweets_data)

        articles = driver.find_elements(By.XPATH, '//article[@role="article" and @data-testid="tweet"]')

        print(f"gevonden articles: {len(articles)} | tweets: {len(tweets_data)}")

        for article in articles:
            try:
                # Datum
                time_elem = article.find_element(By.XPATH, './/time')
                date_str = time_elem.get_attribute("datetime")
                tweet_date = datetime.fromisoformat(date_str.replace("Z", "+00:00")).replace(tzinfo=None)

                # ID + username
                tweet_link = article.find_element(By.XPATH, './/time/..').get_attribute('href')
                tweet_id = tweet_link

                if tweet_id in tweets_ids:
                    continue

                # extra robuuste username check
                if f"/{user}/" not in tweet_link:
                    continue

                # Pinned check
                try:
                    social_context = article.find_element(By.CSS_SELECTOR, "div[data-testid='socialContext']")
                    is_pinned = "Pinned" in social_context.text
                except:
                    is_pinned = False

                # Stopcondities
                if is_pinned and tweet_date.strftime("%Y-%m-%d") < since:
                    continue

                if tweet_date.strftime("%Y-%m-%d") < since:
                    stop_scraping = True
                    break

                # Tekst
                try:
                    text = article.find_element(By.XPATH, './/div[@data-testid="tweetText"]').text
                except:
                    text = ""

                # Stats
                try:
                    stats_group = article.find_element(By.XPATH, './/div[@role="group"]')
                    label = stats_group.get_attribute("aria-label")
                except:
                    label = ""

                def parse_stat(pattern, text):
                    match = re.search(pattern, text.lower())
                    if match:
                        return int(re.sub(r'[^\d]', '', match.group(1)))
                    return 0

                replies = parse_stat(r'(\d[\d\.,]*)\s+replies', label)
                reposts = parse_stat(r'(\d[\d\.,]*)\s+reposts', label)
                likes = parse_stat(r'(\d[\d\.,]*)\s+likes', label)
                bookmarks = parse_stat(r'(\d[\d\.,]*)\s+bookmarks', label)
                views = parse_stat(r'(\d[\d\.,]*)\s+views', label)

                # Opslaan
                tweets_data.append([
                    tweet_date, user, text,
                    replies, reposts, likes, bookmarks, views
                ])
                tweets_ids.add(tweet_id)

                print(f"✔ {tweet_date} | likes: {likes}")

                # Batch refresh
                if len(tweets_ids) % 363 == 0:
                    until = tweet_date.strftime("%Y-%m-%d")
                    print(f"Batch → nieuwe until: {until}")
                    refresh_needed = True
                    break

            except Exception as e:
                print("Fout:", e)
                continue

        # 🔁 Batch refresh
        if refresh_needed:
            retry += 1
            twitter_handle(username=user, since=since, until=until)
            continue

        # 🔁 Stagnatie check
        tweets_after = len(tweets_data)

        if tweets_after == tweets_before:
            retry += 1
            print(f"⚠ Geen nieuwe tweets (retry {retry})")

            if retry >= MAX_RETRY:
                print("⛔ Max retries bereikt → stoppen")
                break

            if tweets_data:
                last_tweet_date = tweets_data[-1][0]
                until = last_tweet_date.strftime("%Y-%m-%d")
                twitter_handle(username=user, since=since, until=until)

            continue
        else:
            retry = 0  # reset als er wel nieuwe tweets zijn

        # Scroll
        human_scroll(driver, total_scroll=2400, step=250, pause=0.2)
        time.sleep(scroll_pause)

    # 💾 Opslaan per user
    if tweets_data:
        df = pd.DataFrame(tweets_data, columns=[
            "date", "username", "text",
            "replies", "reposts", "likes", "bookmarks", "views"
        ])

        df["date"] = pd.to_datetime(df["date"], errors="coerce")

        os.makedirs(f"../../raw/tweets/{user}", exist_ok=True)
        df.to_csv(f"../../raw/tweets/{user}/{user}_tweets.csv", index=False, mode="a")

        print(f"✅ {len(df)} tweets opgeslagen voor {user}")
    else:
        print(f"❌ Geen tweets gevonden voor {user}")

driver.close()

print("🎉 Klaar!")


--- Start scraping elonmusk ---
Cookies geaccepteerd.
gevonden articles: 8 | tweets: 0
✔ 2026-01-21 23:08:33 | likes: 27530
✔ 2026-01-21 23:07:16 | likes: 3950
✔ 2026-01-21 18:55:49 | likes: 30851
✔ 2026-01-21 18:15:19 | likes: 88733
✔ 2026-01-21 12:07:42 | likes: 46924
✔ 2026-01-21 12:05:18 | likes: 6472
✔ 2026-01-21 11:57:15 | likes: 14270
✔ 2026-01-21 09:56:12 | likes: 3029
gevonden articles: 18 | tweets: 8
✔ 2026-01-21 09:51:49 | likes: 147792
✔ 2026-01-21 09:50:02 | likes: 469
✔ 2026-01-21 09:48:30 | likes: 4682
✔ 2026-01-21 09:45:13 | likes: 3421
✔ 2026-01-21 09:39:30 | likes: 50167
✔ 2026-01-21 09:33:05 | likes: 8431
✔ 2026-01-21 09:26:41 | likes: 787
✔ 2026-01-21 09:20:02 | likes: 3253
✔ 2026-01-21 09:13:44 | likes: 616
✔ 2026-01-21 08:57:00 | likes: 199745
✔ 2026-01-21 08:20:54 | likes: 19277
✔ 2026-01-21 08:18:19 | likes: 42106
gevonden articles: 11 | tweets: 20
⚠ Geen nieuwe tweets (retry 1)
Geen cookie-wall gevonden of al geaccepteerd.
gevonden articles: 9 | tweets: 20
✔ 2